<a href="https://colab.research.google.com/github/MeenaShree10/Financial-fraud-detection-prj2/blob/main/Fraud_detection_weighted_loss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
2+2

4

In [2]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.8 MB/s eta 0:00:00


In [3]:
from torch_geometric.datasets import EllipticBitcoinDataset

print("Downloading the Elliptic Bitcoin Dataset...")
dataset = EllipticBitcoinDataset(root='./data')
print("Download complete!")

# Let's peek inside the dataset to see how massive it is
data = dataset[0]
print(f"Number of transactions (nodes): {data.num_nodes}")
print(f"Number of relationships (edges): {data.num_edges}")

Processing...


Download complete!
Number of transactions (nodes): 203769
Number of relationships (edges): 234355


Done!


In [4]:
import torch

# The dataset stores the answers (labels) in a variable called 'y'
labels, counts = torch.unique(data.y, return_counts=True)

print("Breaking down the transactions by category:")
for label, count in zip(labels, counts):
    print(f"Category {label.item()}: {count.item()} transactions")

Breaking down the transactions by category:
Category 0: 42019 transactions
Category 1: 4545 transactions
Category 2: 157205 transactions


In [5]:
from torch.nn import Linear
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GCN(torch.nn.Module):
    def __init__(self):
        super(GCN, self).__init__()
        # The first layer takes the 166 features of each transaction and compresses them
        self.conv1 = GCNConv(dataset.num_node_features, 64)

        # The second layer takes those 64 compressed features and outputs our final decision (Licit or Illicit)
        self.conv2 = GCNConv(64, dataset.num_classes)

    def forward(self, x, edge_index):
        # Step 1: Pass data through the first layer
        x = self.conv1(x, edge_index)

        # We apply a ReLU activation function to help it learn complex, non-linear patterns
        x = F.relu(x)

        # We apply "Dropout" to randomly turn off parts of the network so it doesn't just memorize the data
        x = F.dropout(x, p=0.5, training=self.training)

        # Step 2: Pass through the final layer to get our prediction
        x = self.conv2(x, edge_index)

        return x

# Instantiate the model to bring it to life
model = GCN()
print("Model Architecture:")
print(model)

Model Architecture:
GCN(
  (conv1): GCNConv(165, 64)
  (conv2): GCNConv(64, 2)
)


In [6]:
import torch

# The Adam optimizer is highly efficient at iteratively minimizing our error
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

# CrossEntropyLoss is the standard scorekeeper for classification tasks
# Calculate the ratio of legitimate to fraudulent transactions
weight_for_fraud = 42019.0 / 4545.0

# Apply a weight of 1.0 to normal transactions, and ~9.2 to fraud
class_weights = torch.tensor([1.0, weight_for_fraud])

# Update the scorekeeper to use these mathematical weights
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

# Grab our single massive graph
data = dataset[0]

def train():
    model.train()          # Set the model to "learning mode"
    optimizer.zero_grad()  # Clear out the math from the previous step

    # 1. Forward Pass: Ask the model to guess who is committing fraud
    out = model(data.x, data.edge_index)

    # 2. Calculate the Error: We use 'train_mask' to ONLY score the known transactions
    loss = criterion(out[data.train_mask], data.y[data.train_mask])

    # 3. Backward Pass: Calculate exactly how to adjust the matrix weights to fix the errors
    loss.backward()

    # 4. Update the weights
    optimizer.step()

    return loss.item()

print("Starting the training process...")
# We will loop through the entire dataset 100 times (called 'epochs')
for epoch in range(101):
    loss = train()
    # Print the score every 10 rounds so we can watch it learn
    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Error (Loss): {loss:.4f}')

print("Training complete!")

Starting the training process...
Epoch: 000, Error (Loss): 1.7149
Epoch: 010, Error (Loss): 0.4205
Epoch: 020, Error (Loss): 0.3323
Epoch: 030, Error (Loss): 0.3009
Epoch: 040, Error (Loss): 0.2731
Epoch: 050, Error (Loss): 0.2556
Epoch: 060, Error (Loss): 0.2416
Epoch: 070, Error (Loss): 0.2269
Epoch: 080, Error (Loss): 0.2219
Epoch: 090, Error (Loss): 0.2113
Epoch: 100, Error (Loss): 0.2056
Training complete!


In [7]:
from sklearn.metrics import f1_score

def test():
    model.eval() # Put the model into testing mode (turns off dropout)

    # We use torch.no_grad() because we don't need to calculate gradients for updating weights anymore
    with torch.no_grad():
        out = model(data.x, data.edge_index)

    # The model outputs two probabilities per transaction [Licit, Illicit].
    # We use argmax to pick the category with the highest probability as our final guess.
    pred = out.argmax(dim=1)

    # We apply the 'test_mask' to isolate ONLY the transactions the model has never seen
    test_labels = data.y[data.test_mask]
    test_preds = pred[data.test_mask]

    # Calculate the F1 Score specifically for the minority "Fraud" class (Category 1)
    f1 = f1_score(test_labels.cpu(), test_preds.cpu(), pos_label=1, average='binary')

    # Calculate overall accuracy just as a baseline comparison
    correct = (test_preds == test_labels).sum().item()
    accuracy = correct / data.test_mask.sum().item()

    return accuracy, f1

acc, f1 = test()
print("--- Final Exam Results ---")
print(f"Overall Accuracy: {acc * 100:.2f}%")
print(f"Fraud Detection F1-Score: {f1:.4f}")

--- Final Exam Results ---
Overall Accuracy: 88.11%
Fraud Detection F1-Score: 0.4001
